In [1]:
import numpy as np
import json
import plotly.graph_objects as go

In [2]:
def pose_trace(pose):
    # Unpack pose into rotation matrix R and translation vector t
    R, t = pose
    t = np.array(t)  # Ensure t is a numpy array
    
    # Define arrow colors for each axis (RGB)
    colors = ['red', 'green', 'blue']
    
    # Define the unit vectors from the columns of R
    axis_vectors = [R[:, 0], R[:, 1], R[:, 2]]
    
    # Create traces for each axis (X, Y, Z)
    traces = []
    for i, vec in enumerate(axis_vectors):
        arrow_start = t
        arrow_end = t + vec  # Arrow points in the direction of the column of R
        
        # Create an arrow trace for the axis
        trace = go.Scatter3d(
            x=[arrow_start[0], arrow_end[0]],
            y=[arrow_start[1], arrow_end[1]],
            z=[arrow_start[2], arrow_end[2]],
            mode='lines+markers',
            marker=dict(size=4),
            line=dict(color=colors[i], width=5),
            showlegend=False
        )
        traces.append(trace)
    
    return traces

def pose_traces(pose_list):
    all_traces = []

    for pose in pose_list:
        traces = pose_trace(pose)
        all_traces.extend(traces)  
    
    return all_traces

In [3]:
RUN_NAME = 'unreal_moon_cv_fly_1'

In [4]:
transforms_path = f'../data/AirSim/{RUN_NAME}/transforms_colmap.json'
with open(transforms_path, 'r') as f:
    transforms = json.load(f)

poses = []
for frame in transforms['frames']:
    T = np.array(frame['transform_matrix'])
    R, t = T[:3, :3], 5.0 * T[:3, 3]
    poses.append((R, t))

colmap_poses = pose_traces(poses)

In [ ]:
# Create and show the plot with all the traces
fig = go.Figure(data=colmap_poses)
fig.update_layout(height=900, width=1600, scene=dict(aspectmode='data'))
fig.show()

In [6]:
fig.write_html('colmap_poses.html')

In [7]:
transforms_path = f'../data/AirSim/{RUN_NAME}/transforms_airsim.json'
with open(transforms_path, 'r') as f:
    transforms = json.load(f)

poses = []
for frame in transforms['frames']:
    T = np.array(frame['transform_matrix'])
    R, t = T[:3, :3], T[:3, 3]
    poses.append((R, t))

airsim_poses = pose_traces(poses)

In [ ]:
# Create and show the plot with all the traces
fig = go.Figure(data=airsim_poses)
fig.update_layout(height=900, width=1600, scene=dict(aspectmode='data'))
fig.show()

In [9]:
fig.write_html("airsim_poses.html")

In [ ]:
def generate_spiral(center, num_rings, radii, heights, num_points):
    """General spiral of camera poses in nerfstudio coordinates"""
    poses = []
    transforms = []  # camera transform matrices for nerfstudio

    for i in range(num_rings):
        r = radii[i]
        z = heights[i]
        n = num_points[i]

        for j in range(n):
            # AirSim coordinates
            angle = 2*np.pi*j/n
            x = r*np.cos(angle)
            y = r*np.sin(angle)
            offset = np.array([x, y, -z])
            position = center + offset

            vx = -offset
            vx = vx / np.linalg.norm(vx)
            d = np.linalg.norm(vx[:2])
            unit_z = vx[2]
            vz = np.array([-(unit_z/d)*vx[0], -(unit_z/d)*vx[1], d])
            vy = np.cross(vz, vx)
            airsim_R = np.vstack((vx, vy, vz)).T
            q = R_to_quat(airsim_R)
            print(f"airsim R:\n{airsim_R}")

            poses.append((position, q))

            # Nerfstudio coordinates
            c2w = np.eye(4)
            ns_pos = np.array([y, x, z])  # +Z is back and away from camera (opposite look-direction)
            vz = ns_pos / np.linalg.norm(ns_pos)  # normalize
            d = np.linalg.norm(vz[:2])  # distance in x-y plane
            unit_z = vz[2]              # z component of unit vector
            vy = np.array([-(unit_z/d)*vz[0], -(unit_z/d)*vz[1], d])  # y is in x-y plane
            vx = np.cross(vy, vz)
            c2w[:3, :3] = np.vstack((vx, vy, vz)).T
            c2w[:3, 3] = ns_pos
            print(f"nerfstudio R:\n{c2w[:3, :3]}")
            transforms.append(c2w)

            NED_2_NERFSTUDIO = np.array([[0, 0, -1],
                                 [1, 0, 0],
                                 [0, -1, 0]])  # or inverse?


    return poses, transforms